# Adult / Census Income — Data Preparation for Fairness Analysis

Each step is **action + justification** (methodology for the fairness audit).

## Setup

In [ ]:
from pathlib import Path

import pandas as pd

# Resolve data dir whether the kernel cwd is `data/` or the project root
_cwd = Path.cwd().resolve()
if (_cwd / "adult.csv").exists():
    DATA_DIR = _cwd
elif (_cwd / "data" / "adult.csv").exists():
    DATA_DIR = _cwd / "data"
else:
    DATA_DIR = Path("data").resolve()

RAW_CSV = DATA_DIR / "adult.csv"
CLEAN_CSV = DATA_DIR / "adult_clean.csv"

PROTECTED_ATTRIBUTE = "gender"  # race audited in a later pass
OUTCOME = "income"

print(f"DATA_DIR: {DATA_DIR}")
print(f"RAW_CSV exists: {RAW_CSV.exists()}")

## Step 1 — Load and inspect

**Justification:** Inspecting before altering is good practice; cleaning choices cannot be justified if made blind.

In [ ]:
df = pd.read_csv(RAW_CSV)

print(f"df.shape: {df.shape}")
display(df.head())
df.info()
display(df.describe())

## Step 2 — Column roles

**Justification:** Document which columns play which role in the fairness analysis so the reader knows what is audited against what.

In [ ]:
roles = {
    "x": "Row identifier (no person-level meaning)",
    "age": "Numeric feature — age in years",
    "workclass": "Categorical feature — type of employer",
    "education": "Categorical feature — highest education level",
    "marital-status": "Categorical feature — marital status",
    "relationship": "Categorical feature — household relationship",
    "race": "Protected attribute (secondary audit, later)",
    "gender": "Protected attribute (primary audit)",
    "hours-per-week": "Numeric feature — hours worked per week",
    "income": "Outcome — income band (<=50K / >50K)",
}

for col in df.columns:
    print(f"  • {col}: {roles.get(col, '(see dataset docs)')}")

print(f"\nKey columns for this project:")
print(f"  Protected attribute → {PROTECTED_ATTRIBUTE}")
print(f"  Outcome             → {OUTCOME}")

## Step 3 — Find missing values encoded as `?`

**Justification:** This dataset hides missingness as the string `'?'`, so a normal missing-value check misses them. Counting `'?'` shows the data was not assumed clean.

In [ ]:
question_counts = (df.astype(str) == "?").sum()
print(question_counts)
print(f"\nTotal cells with '?': {int(question_counts.sum())}")
print(f"Rows containing at least one '?': {(df.astype(str) == '?').any(axis=1).sum()}")

## Step 4 — Handle missing values (delete vs impute)

**Decision:** DELETE rows containing `'?'`.

**Alternative considered:** IMPUTE (e.g. mode of workclass).

**Justification:** Deletion loses a small share of records (under 6%). Imputation was rejected because inventing values risks distorting exactly the under-recorded groups a fairness audit exists to protect (cf. Pagano et al.).

In [ ]:
n_before = len(df)
mask_missing = (df.astype(str) == "?").any(axis=1)
n_dropped = int(mask_missing.sum())
cleaned = df.loc[~mask_missing].copy()
n_after = len(cleaned)
pct_lost = 100.0 * n_dropped / n_before

print(
    f"Deletion loses {n_dropped:,} of {n_before:,} records "
    f"({pct_lost:.2f}%: {n_before:,} → {n_after:,})"
)
print(f"Shape after deletion: {cleaned.shape}")
df = cleaned

## Step 5 — Remove genuinely useless columns

**Decision:** Drop `x`.

**Justification:** `x` is only a row number — an identifier with no predictive or analytical value. Keeping it could confuse a model later.

In [ ]:
df = df.drop(columns=["x"])
print(f"Shape after dropping x: {df.shape}")
print(f"Columns: {list(df.columns)}")

## Step 6 — Check outcome and protected columns

**Justification:** Fairness calculations depend on these columns being consistent, so verifying labels (and stripping stray spaces) is necessary preparation.

In [ ]:
for col in (OUTCOME, PROTECTED_ATTRIBUTE, "race"):
    if col in df.columns and df[col].dtype == object:
        df[col] = df[col].astype(str).str.strip()
    print(f"{col} value counts:")
    print(df[col].value_counts())
    print(f"{col} unique: {sorted(df[col].unique())}\n")

## Step 7 — Confirm final state

**Justification:** Verify cleaning did what was intended rather than assuming it — closing the loop.

In [ ]:
remaining_q = int((df.astype(str) == "?").sum().sum())
print(f"Final df.shape: {df.shape}  (expected (46043, 9))")
print(f"Remaining '?': {remaining_q}")
assert remaining_q == 0, "Expected no '?' after cleaning"
assert df.shape == (46043, 9), f"Expected (46043, 9), got {df.shape}"
print("Checks passed.")

## Step 8 — What we did NOT do (yet), and why

Scaling numeric features and encoding categorical columns are deferred to the modelling stage. Current disparity / selection-rate measurement operates on raw categories; encoding is only required once a classifier is trained.

## Optional — Save cleaned dataset

In [ ]:
df.to_csv(CLEAN_CSV, index=False)
print(f"Wrote {CLEAN_CSV} ({df.shape[0]:,} rows × {df.shape[1]} cols)")